## GoldWork Incremental
Incremental Gold processing plus latest and timestamped Volume snapshots

## Step-1 Imports and steup
This cell imports the req helpers,switchs o the right catalog,makes sure the gold schema exists and creates
- a `gold_run_id`
- a run date string
- a run timestamp string 


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog  databricks_ecom_project")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("Current Gold Run ID:",gold_run_id)
print("run Timestamp Folder:",run_ts_str)

## step-2 Gold control table
This table stroes the latest Gold processing state

it tells Gold:
- which silver data was processed last time
- how many gold rows were merged in the last run

In [0]:
spark.sql("""
DROP TABLE IF EXISTS databricks_ecom_project.gold_schema.processing_control
""")

In [0]:
from delta.tables import DeltaTable
from datetime import datetime, UTC

spark.sql("""
CREATE TABLE IF NOT EXISTS databricks_ecom_project.gold_schema.processing_control (
    layer STRING,
    entity_name STRING,
    last_processed_silver_run_id STRING,
    last_processed_silver_run_ts TIMESTAMP,
    rows_merged BIGINT,
    run_status STRING,
    gold_run_id STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

## Step-3 Helper Functions
- `upsert_to_gold()` merges data into Gold current-state tables
- `get_last_processed_silver_ts()`
- `upsert_gold_control()` updates Gold control after a successful run

In [0]:
def upsert_to_gold(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark,target_table)
        (dt.alias("target")
           .merge(df_source.alias("source"),f"target.{target_table} = source.{join_key}")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
         )
    else:
         df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name):
    ctrl = (
        spark.table("databricks_ecom_project.gold_schema.processing_control")
        .filter(
            (F.col("layer") == 'gold') & 
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == 'SUCCESS')
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(
    entity_name,
    last_processed_silver_run_id,
    last_processed_silver_run_ts,
    rows_merged,
    gold_run_id
):
    ctrl_df = spark.createDataFrame(
        [
            (
                "gold",
                entity_name,
                last_processed_silver_run_id,
                last_processed_silver_run_ts,
                int(rows_merged),
                "SUCCESS",
                gold_run_id,
                datetime.now(UTC).replace(tzinfo=None)
            )
        ],
        schema="""
            layer STRING,
            entity_name STRING,
            last_processed_silver_run_id STRING,
            last_processed_silver_run_ts TIMESTAMP,
            rows_merged BIGINT,
            run_status STRING,
            gold_run_id STRING,
            updated_at TIMESTAMP
        """
    )

    dt = DeltaTable.forName(
        spark,
        "databricks_ecom_project.gold_schema.processing_control"
    )

    (
        dt.alias("target")
        .merge(
            ctrl_df.alias("source"),
            "target.layer = source.layer AND target.entity_name = source.entity_name"
        )
        .whenMatchedUpdate(set={
            "last_processed_silver_run_id": "source.last_processed_silver_run_id",
            "last_processed_silver_run_ts": "source.last_processed_silver_run_ts",
            "rows_merged": "source.rows_merged",
            "run_status": "source.run_status",
            "gold_run_id": "source.gold_run_id",
            "updated_at": "source.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

## Step-4 Read changed Silver rows only
This cell reads the full silver current-state tables but filters out only the rows that changed since the last gold run.
This s the starting point for Gold incremental processing


In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")
print("last Processed silver Timestamp for Gold =", last_gold_ts)
silver_orders_current = spark.read.table("databricks_ecom_project.silver_schema.orders_transformed")
silver_products_current = spark.read.table("databricks_ecom_project.silver_schema.products_transformed")
silver_payments_current = spark.read.table("databricks_ecom_project.silver_schema.payments_transformed")

if last_gold_ts is None :
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("updated_at") > F.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders  = {changed_orders_count}")
print(f"Number of changed products  =  {changed_products_count}")
print(f"Number of changed payments =  {changed_payments_count}")


###  Step-5 Find impacted order IDs
Gold bulit at order grain,so if anything changes in orders,products,or payments,we identify which `order_id` values are impacted

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()

impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
          F.col("p.product_id") == F.col("o.product_id"),
          "inner"
          )
    .select(F.col("o.order_id"))
    .distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
    
)

print("impacted_order_ids =",impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))





### Step-6 - Bulid Gold delta for impacted orders
This Cell joins impacted orders with the current Silver products and payments tables,derives bussines columns and bulids the Gold delta that will be merged into the Gold current state table

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        F.col("o.order_id") == F.col("py.order_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("py.processed_at").cast("timestamp")
        ).alias("gold_update_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        F.when(
            F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
         .when(F.col("payment_completion_ratio") == 0, "Unpaid")
         .when(F.col("payment_completion_ratio") == 1, "Paid")
         .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
         .when(F.col("payment_completion_ratio") > 1, "Overpaid")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))
    .withColumn("gold_run_id", F.lit(gold_run_id))
)


gold_delta_count = gold_delta.count()
print("gold_delta_rows =", gold_delta_count)
display(gold_delta)

### Step-7 - Merge Gold current-state table
If Gold delta contains rows,this cell merges them into `gold_schema.orders_information`.

In [0]:

if gold_delta_count > 0:
    upsert_to_gold(
        gold_delta,
        "databricks_ecom_project.gold_schema.orders_information",
        "order_id"
    )
else:
    print("No new rows to insert in gold table")

In [0]:
%sql
select * from databricks_ecom_project.gold_schema.orders_information;

### step -8 Maintain Gold SCD Type 2 history
This cell updates the SCD2 history table.
if the current gold row changes,the old version is closed(is_current = false) and new current version is inserted.

In [0]:

if not spark.catalog.tableExists("databricks_ecom_project.gold_schema.orders_information_scd2"):
    spark.sql("""
          create table databricks_ecom_project.gold_schema.orders_information_scd2
          using delta
          as
          select *,
                 cast(null as timestamp) as valid_from_ts,
                 cast(null as timestamp) as valid_to_ts,
                 true as is_current
                 from databricks_ecom_project.gold_schema.orders_information
                 where 1 =0
          """)
if gold_delta.count() >0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into databricks_ecom_project.gold_schema.orders_information_scd2 t
              using gold_delta_view s
              on t.order_id = s.order_id and t.is_current = true
              when matched and (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_status <=> s.payment_status) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price))
                
            then update set
                `is_current` = false,
                `valid_to_ts` = s.gold_update_ts
             """)
    
    spark.sql("""
        INSERT INTO databricks_ecom_project.gold_schema.orders_information_scd2
        select
            s.*,
            s.gold_update_ts AS valid_from_ts,
            CAST(NULL AS TIMESTAMP) AS valid_to_ts,
            TRUE AS is_current
        FROM gold_delta_view s
        LEFT JOIN databricks_ecom_project.gold_schema.orders_information_scd2 t
          ON s.order_id = t.order_id
         AND t.is_current = true
        WHERE t.order_id IS NULL
           OR (
                NOT (t.order_status <=> s.order_status) OR
                NOT (t.order_amount <=> s.order_amount) OR
                NOT (t.paid_amount <=> s.paid_amount) OR
                NOT (t.payment_id <=> s.payment_id) OR
                NOT (t.category <=> s.category) OR
                NOT (t.product_name <=> s.product_name) OR
                NOT (t.product_price <=> s.product_price)
           )
    """)
     

### Step9 - Update category level Gold aggregation
This cell recalculates category-level bussines metrics only for categories impacted in the current run,then merges them into the category performance gold table

In [0]:

if gold_delta_count > 0:
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(
            F.col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table("databricks_ecom_project.gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                 .otherwise(F.lit(0.0))
            ).alias("gross_merchandise_value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                 .otherwise(F.lit(0.0))
            ).alias("total_amount_paid"),
            F.avg(F.col("payment_completion_ratio")).alias("avg_payment_completion_ratio"),
            (
                F.sum(F.when(F.col("payment_status") == "FAILED", 1).otherwise(0)) / F.count("*")
            ).alias("payment_failure_rate")
        )
    )

    upsert_to_gold(
        category_perf_delta,
        "databricks_ecom_project.gold_schema.category_performance",
        "category"
    )

else:
    print("No impacted categories to update in gold category_performance.")
     

In [0]:
%sql
select * from databricks_ecom_project.gold_schema.category_performance;

In [0]:
if gold_delta_count > 0:
    impacted_payment_status = (
        gold_delta
        .select("payment_status")
        .filter(F.col("payment_status").isNotNull())
        .distinct()
    )

    payment_status_delta = (
        spark.read.table("databricks_ecom_project.gold_schema.orders_information")
        .join(impacted_payment_status, "payment_status", "inner")
        .groupBy("payment_status")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.countDistinct("payment_id").alias("total_payments"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                 .otherwise(F.lit(0.0))
            ).alias("total_paid_amount"),
            F.avg("payment_completion_ratio").alias("avg_payment_completion_ratio"),
            F.count("*").alias("record_count")
        )
        .withColumn("gold_update_ts", F.current_timestamp())
    )

    upsert_to_gold(
        payment_status_delta,
        "databricks_ecom_project.gold_schema.payment_status_summary",
        "payment_status"
    )

else:
    print("No impacted payment statuses to update payment_status_summary.")

In [0]:
%sql 
select * from databricks_ecom_project.gold_schema.payment_status_summary;

### Step-10 Publish Gold snapshots to Volume
This cell writes two kinds of gold outputs to a databricks Volumes
- latest snapshot overwritten evry successful run
- timestamped historical snapshot - a new folder for each successful run

This is useful for auidt,roolback

In [0]:

spark.sql("create volume if not exists databricks_ecom_project.gold_schema.gold_snapshots_vol")

In [0]:
latest_orders_path =(
    "/Volumes/databricks_ecom_project/gold_schema/gold_snapshots_vol/gold_latest/orders_information"

)

latest_category_path =(
    "/Volumes/databricks_ecom_project/gold_schema/gold_snapshots_vol/gold_latest/category_performance"
)

historical_orders_path = f"/Volumes/databricks_ecom_project/gold_schema/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts = {run_ts_str}"

historical_category_path = f"/Volumes/databricks_ecom_project/gold_schema/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts = {run_ts_str}"

spark.read.table("databricks_ecom_project.gold_schema.orders_information") \
    .write.mode("overwrite").format("parquet").save(latest_orders_path)

spark.read.table("databricks_ecom_project.gold_schema.category_performance") \
    .write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("databricks_ecom_project.gold_schema.orders_information") \
    .write.mode("overwrite").format("parquet").save(historical_orders_path)

spark.read.table("databricks_ecom_project.gold_schema.category_performance") \
    .write.mode("overwrite").format("parquet").save(historical_category_path)

print("latest Orders path:", latest_orders_path)
print("latest Category path:", latest_category_path)
print("historical Orders path:", historical_orders_path)
print("historical Category path:", historical_category_path)

### update gold control table
This final cell updates the gold control table using latest silver processing metadata and display the control table for validation

In [0]:
latest_silver_ts = (
    silver_orders_current
    .agg(F.max("bronze_ingested_at").alias("mx"))
    .collect()[0]["mx"]
)

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

gold_delta_count = gold_delta.count()

upsert_gold_control(
    "orders_information",
    latest_silver_run_id,
    latest_silver_ts,
    gold_delta_count,
    gold_run_id
)

display(
    spark.table("databricks_ecom_project.gold_schema.processing_control")
)

In [0]:
%sql
select * from databricks_ecom_project.gold_schema.orders_information_scd2 where order_id = 200002;